# Day 025 | Seaborn — Statistical Visualization

## 100 Days of ML for Geotechnical Engineering
**Phase:** Data Science Essentials

**Author:** Ripon Chandra Malo  
**Date:** YYYY-MM-DD  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| `sns.histplot()` | Distribution of a variable | Distribution of N-values across a site |
| `sns.boxplot()` | Spread and outliers per group | N-value range for each soil type |
| `sns.violinplot()` | Distribution shape per group | Shear wave velocity distribution by soil type |
| `sns.heatmap()` | Color-coded matrix | Correlation matrix of soil properties |
| `sns.pairplot()` | All pairwise scatter plots | Relationships between all geotechnical parameters |
| `sns.regplot()` | Scatter with regression line | N-value vs shear wave velocity trend |
| `set_theme()` | Global styling | Publication-quality geotechnical plots |

---

## Our Geotechnical Problem

A geotechnical investigation collected data from **50 soil samples** across a construction site. Each sample has depth, SPT N-value, shear wave velocity (Vs), water content, unit weight, void ratio, fines content, and soil type classification.

The engineer needs to:
- Understand the **distribution** of each soil property  
- Compare properties across **soil types** (Clay, Sand, Silt, Gravel)  
- Identify **correlations** between parameters  
- Visualize the **N-value vs Vs** relationship for empirical correlation development

Seaborn makes statistical visualization effortless.

---

## Part 1: Generating Synthetic Geotechnical Data

We create a realistic dataset of 50 soil samples with physically consistent properties. The relationships between parameters (e.g., clay has higher water content, sand has higher N-values at depth) reflect real geotechnical behavior.

In [ ]:
# ─────────────────────────────────────────────────────
# GENERATING SYNTHETIC GEOTECHNICAL SITE DATA
# ─────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

np.random.seed(42)

n_samples = 50
soil_types = np.random.choice(['Clay', 'Sand', 'Silt', 'Gravel'],
                               size=n_samples, p=[0.30, 0.35, 0.20, 0.15])
depth = np.round(np.random.uniform(1.0, 25.0, n_samples), 1)

# Generate physically consistent properties per soil type
N_value = np.zeros(n_samples)
Vs = np.zeros(n_samples)
water_content = np.zeros(n_samples)
unit_weight = np.zeros(n_samples)
void_ratio = np.zeros(n_samples)
fines_content = np.zeros(n_samples)

for i in range(n_samples):
    d = depth[i]
    if soil_types[i] == 'Clay':
        N_value[i] = max(2, int(3 + 1.2 * d + np.random.normal(0, 4)))
        Vs[i] = 100 + 8 * d + np.random.normal(0, 20)
        water_content[i] = 35 - 0.5 * d + np.random.normal(0, 5)
        unit_weight[i] = 17.5 + 0.08 * d + np.random.normal(0, 0.5)
        void_ratio[i] = 0.9 - 0.015 * d + np.random.normal(0, 0.08)
        fines_content[i] = 85 + np.random.normal(0, 5)
    elif soil_types[i] == 'Sand':
        N_value[i] = max(3, int(5 + 2.0 * d + np.random.normal(0, 5)))
        Vs[i] = 130 + 12 * d + np.random.normal(0, 25)
        water_content[i] = 18 - 0.3 * d + np.random.normal(0, 3)
        unit_weight[i] = 18.5 + 0.1 * d + np.random.normal(0, 0.4)
        void_ratio[i] = 0.65 - 0.01 * d + np.random.normal(0, 0.06)
        fines_content[i] = 12 + np.random.normal(0, 5)
    elif soil_types[i] == 'Silt':
        N_value[i] = max(2, int(4 + 1.5 * d + np.random.normal(0, 4)))
        Vs[i] = 110 + 10 * d + np.random.normal(0, 22)
        water_content[i] = 28 - 0.4 * d + np.random.normal(0, 4)
        unit_weight[i] = 18.0 + 0.09 * d + np.random.normal(0, 0.5)
        void_ratio[i] = 0.75 - 0.012 * d + np.random.normal(0, 0.07)
        fines_content[i] = 70 + np.random.normal(0, 8)
    else:  # Gravel
        N_value[i] = max(5, int(10 + 2.5 * d + np.random.normal(0, 6)))
        Vs[i] = 160 + 14 * d + np.random.normal(0, 30)
        water_content[i] = 12 - 0.2 * d + np.random.normal(0, 2)
        unit_weight[i] = 19.5 + 0.1 * d + np.random.normal(0, 0.4)
        void_ratio[i] = 0.50 - 0.008 * d + np.random.normal(0, 0.05)
        fines_content[i] = 5 + np.random.normal(0, 3)

df = pd.DataFrame({
    'depth_m': depth,
    'N_value': N_value.astype(int),
    'Vs_m_s': np.round(Vs, 1),
    'water_content': np.round(np.clip(water_content, 5, 55), 1),
    'unit_weight': np.round(np.clip(unit_weight, 15, 22), 1),
    'void_ratio': np.round(np.clip(void_ratio, 0.25, 1.2), 2),
    'fines_content': np.round(np.clip(fines_content, 0, 100), 1),
    'soil_type': soil_types
})

print(f"Dataset: {len(df)} soil samples")
print(f"Soil types: {df['soil_type'].value_counts().to_dict()}")
print()
print(df.describe().round(1).to_string())
print()
df.head(10)

---

## Part 2: Setting the Theme and `histplot()`

Seaborn's `set_theme()` controls the global look. `histplot()` shows how a variable is distributed — critical for checking if your geotechnical data is normal, skewed, or multimodal.

In [ ]:
# ─────────────────────────────────────────────────────
# THEME SETUP AND HISTPLOT
# ─────────────────────────────────────────────────────

# Set a clean theme for all plots
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Histogram of N-values
sns.histplot(df['N_value'], bins=15, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Distribution of SPT N-values')
axes[0].set_xlabel('N-value (blows/30cm)')
axes[0].axvline(df['N_value'].mean(), color='red', linestyle='--',
                label=f"Mean = {df['N_value'].mean():.0f}")
axes[0].legend()

# Histogram of Vs
sns.histplot(df['Vs_m_s'], bins=15, kde=True, color='darkorange', ax=axes[1])
axes[1].set_title('Distribution of Shear Wave Velocity')
axes[1].set_xlabel('Vs (m/s)')

# Histogram by soil type (stacked)
sns.histplot(data=df, x='water_content', hue='soil_type', bins=12,
             multiple='stack', ax=axes[2])
axes[2].set_title('Water Content by Soil Type')
axes[2].set_xlabel('Water Content (%)')

plt.tight_layout()
plt.show()

print("KDE overlay (kde=True) shows the smoothed density estimate.")
print("The stacked histogram reveals which soil types contribute to each bin.")

---

## Part 3: `boxplot()` — Comparing Groups

Box plots show the median, quartiles, and outliers for each group. Perfect for comparing soil properties across soil types or boreholes.

In [ ]:
# ─────────────────────────────────────────────────────
# BOXPLOT — Soil properties by type
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# N-value by soil type
sns.boxplot(data=df, x='soil_type', y='N_value',
            order=['Clay', 'Silt', 'Sand', 'Gravel'],
            palette='Set2', ax=axes[0])
axes[0].set_title('SPT N-value by Soil Type')
axes[0].set_ylabel('N-value (blows/30cm)')

# Water content by soil type
sns.boxplot(data=df, x='soil_type', y='water_content',
            order=['Clay', 'Silt', 'Sand', 'Gravel'],
            palette='Set2', ax=axes[1])
axes[1].set_title('Water Content by Soil Type')
axes[1].set_ylabel('Water Content (%)')

# Void ratio by soil type
sns.boxplot(data=df, x='soil_type', y='void_ratio',
            order=['Clay', 'Silt', 'Sand', 'Gravel'],
            palette='Set2', ax=axes[2])
axes[2].set_title('Void Ratio by Soil Type')
axes[2].set_ylabel('Void Ratio (e)')

plt.tight_layout()
plt.show()

print("Gravel has the highest median N-value (strongest).")
print("Clay has the highest water content and void ratio.")
print("Outliers (diamonds) may indicate unusual layers or test errors.")

---

## Part 4: `violinplot()` — Distribution Shape

Violin plots combine box plots with kernel density estimation to show the full distribution shape. This reveals bimodal distributions or skewness that box plots hide.

In [ ]:
# ─────────────────────────────────────────────────────
# VIOLINPLOT — Distribution shape per soil type
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Vs distribution by soil type
sns.violinplot(data=df, x='soil_type', y='Vs_m_s',
               order=['Clay', 'Silt', 'Sand', 'Gravel'],
               palette='coolwarm', inner='box', ax=axes[0])
axes[0].set_title('Shear Wave Velocity Distribution by Soil Type')
axes[0].set_ylabel('Vs (m/s)')

# Unit weight distribution by soil type
sns.violinplot(data=df, x='soil_type', y='unit_weight',
               order=['Clay', 'Silt', 'Sand', 'Gravel'],
               palette='coolwarm', inner='quartile', ax=axes[1])
axes[1].set_title('Unit Weight Distribution by Soil Type')
axes[1].set_ylabel('Unit Weight (kN/m³)')

plt.tight_layout()
plt.show()

print("inner='box' shows a miniature boxplot inside the violin.")
print("inner='quartile' draws horizontal lines at Q1, median, Q3.")
print("Wide sections = more samples at that value.")

---

## Part 5: `heatmap()` — Correlation Matrix

A correlation heatmap shows how strongly each pair of numerical variables is related. In geotechnics, we expect strong correlations like N-value vs Vs (positive) and void ratio vs unit weight (negative).

In [ ]:
# ─────────────────────────────────────────────────────
# HEATMAP — Correlation matrix of soil properties
# ─────────────────────────────────────────────────────

# Select numeric columns for correlation
numeric_cols = ['depth_m', 'N_value', 'Vs_m_s', 'water_content',
                'unit_weight', 'void_ratio', 'fines_content']
corr = df[numeric_cols].corr().round(2)

# Create a mask for the upper triangle (avoid showing duplicate info)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Correlation Matrix of Soil Properties', fontsize=14, pad=15)

plt.tight_layout()
plt.show()

print("Key correlations:")
print(f"  N_value vs Vs:           r = {corr.loc['N_value', 'Vs_m_s']:.2f} (strong positive — expected)")
print(f"  void_ratio vs unit_wt:   r = {corr.loc['void_ratio', 'unit_weight']:.2f} (negative — denser soil = heavier)")
print(f"  water_content vs fines:  r = {corr.loc['water_content', 'fines_content']:.2f} (fine soils hold more water)")
print(f"  depth vs N_value:        r = {corr.loc['depth_m', 'N_value']:.2f} (N increases with depth/confining stress)")

---

## Part 6: `pairplot()` — All Pairwise Relationships

`pairplot()` creates a matrix of scatter plots for every pair of variables, with histograms on the diagonal. This is the fastest way to explore multivariate relationships in your geotechnical data.

In [ ]:
# ─────────────────────────────────────────────────────
# PAIRPLOT — All pairwise scatter plots
# ─────────────────────────────────────────────────────

# Select key variables to keep the plot readable
pair_cols = ['N_value', 'Vs_m_s', 'water_content', 'void_ratio', 'soil_type']

g = sns.pairplot(df[pair_cols], hue='soil_type',
                 palette='Set2', diag_kind='kde',
                 plot_kws={'alpha': 0.7, 's': 40, 'edgecolor': 'white'},
                 height=2.5)
g.figure.suptitle('Pairwise Relationships of Soil Properties', y=1.02, fontsize=14)

plt.tight_layout()
plt.show()

print("Each dot is one soil sample, colored by soil type.")
print("Diagonal: KDE shows the distribution of each variable per soil type.")
print("Off-diagonal: scatter plots reveal correlations and clustering.")
print("Notice how soil types form distinct clusters — this is what ML exploits.")

---

## Part 7: `regplot()` — Regression Visualization

`regplot()` fits and displays a regression line on a scatter plot. In geotechnics, we use this to develop empirical correlations like the classic **N-value vs Vs** relationship.

In [ ]:
# ─────────────────────────────────────────────────────
# REGPLOT — N-value vs Vs correlation
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Overall N vs Vs regression
sns.regplot(data=df, x='N_value', y='Vs_m_s',
            scatter_kws={'alpha': 0.6, 's': 50, 'edgecolors': 'white'},
            line_kws={'color': 'red', 'linewidth': 2},
            ci=95, ax=axes[0])
axes[0].set_title('N-value vs Shear Wave Velocity (All Soils)')
axes[0].set_xlabel('SPT N-value')
axes[0].set_ylabel('Vs (m/s)')

# Depth vs N-value regression
sns.regplot(data=df, x='depth_m', y='N_value',
            scatter_kws={'alpha': 0.6, 's': 50, 'edgecolors': 'white'},
            line_kws={'color': 'darkgreen', 'linewidth': 2},
            ci=95, ax=axes[1])
axes[1].set_title('Depth vs N-value (Overburden Effect)')
axes[1].set_xlabel('Depth (m)')
axes[1].set_ylabel('SPT N-value')

plt.tight_layout()
plt.show()

# Compute R² for reference
from numpy.polynomial.polynomial import polyfit
r_sq_nv = np.corrcoef(df['N_value'], df['Vs_m_s'])[0, 1] ** 2
r_sq_dn = np.corrcoef(df['depth_m'], df['N_value'])[0, 1] ** 2
print(f"N vs Vs:    R² = {r_sq_nv:.3f} — strong linear trend")
print(f"Depth vs N: R² = {r_sq_dn:.3f} — moderate overburden effect")
print("The 95% CI band shows confidence in the regression line.")

---

## Part 8: Advanced Styling and FacetGrid

Seaborn's `FacetGrid` creates separate panels for each category. Combined with different themes, you can produce publication-quality multi-panel figures.

In [ ]:
# ─────────────────────────────────────────────────────
# FACETGRID AND STYLE CONTEXTS
# ─────────────────────────────────────────────────────

# FacetGrid: N vs Vs regression per soil type
g = sns.FacetGrid(df, col='soil_type', col_wrap=2, height=4,
                  col_order=['Clay', 'Silt', 'Sand', 'Gravel'])
g.map_dataframe(sns.regplot, x='N_value', y='Vs_m_s',
                scatter_kws={'alpha': 0.7, 's': 50},
                line_kws={'color': 'red'})
g.set_axis_labels('SPT N-value', 'Vs (m/s)')
g.set_titles('{col_name}')
g.figure.suptitle('N-value vs Vs by Soil Type', y=1.02, fontsize=14)

plt.tight_layout()
plt.show()

print("FacetGrid creates one panel per soil type.")
print("Each panel has its own regression — the slope may differ by soil type.")
print("This is key for developing soil-type-specific empirical correlations.")

In [ ]:
# ─────────────────────────────────────────────────────
# THEME COMPARISON
# ─────────────────────────────────────────────────────

themes = ['whitegrid', 'darkgrid', 'white', 'ticks']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, theme in zip(axes, themes):
    with sns.axes_style(theme):
        sns.scatterplot(data=df, x='N_value', y='Vs_m_s', hue='soil_type',
                        palette='Set2', ax=ax, legend=False)
        ax.set_title(f"style='{theme}'")
        ax.set_xlabel('N-value')
        ax.set_ylabel('Vs (m/s)')

plt.suptitle('Seaborn Theme Comparison', fontsize=14)
plt.tight_layout()
plt.show()

print("'whitegrid' is best for data-heavy plots (easy to read values).")
print("'ticks' is cleanest for publication figures.")
print("Choose based on your audience — reports vs presentations.")

---

## Part 9: Complete Application — Geotechnical Site Characterization Dashboard

Let us combine all Seaborn techniques into a comprehensive visualization of our site investigation data.

In [ ]:
# ══════════════════════════════════════════════════════
# COMPLETE: GEOTECHNICAL SITE CHARACTERIZATION
# ══════════════════════════════════════════════════════

sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.0)

fig = plt.figure(figsize=(16, 14))
fig.suptitle('GEOTECHNICAL SITE CHARACTERIZATION — 50 SOIL SAMPLES',
             fontsize=16, fontweight='bold', y=0.98)

# Panel 1: N-value distribution by soil type
ax1 = fig.add_subplot(3, 3, 1)
sns.boxplot(data=df, x='soil_type', y='N_value',
            order=['Clay', 'Silt', 'Sand', 'Gravel'], ax=ax1)
ax1.set_title('N-value by Soil Type')
ax1.set_xlabel('')

# Panel 2: Vs distribution
ax2 = fig.add_subplot(3, 3, 2)
sns.histplot(df['Vs_m_s'], bins=12, kde=True, color='steelblue', ax=ax2)
ax2.set_title('Vs Distribution')
ax2.set_xlabel('Vs (m/s)')

# Panel 3: Water content violin
ax3 = fig.add_subplot(3, 3, 3)
sns.violinplot(data=df, x='soil_type', y='water_content',
               order=['Clay', 'Silt', 'Sand', 'Gravel'],
               inner='box', ax=ax3)
ax3.set_title('Water Content Distribution')
ax3.set_xlabel('')

# Panel 4: N vs Vs regression
ax4 = fig.add_subplot(3, 3, 4)
sns.regplot(data=df, x='N_value', y='Vs_m_s', ci=95,
            scatter_kws={'alpha': 0.5, 's': 30}, ax=ax4)
r_sq = np.corrcoef(df['N_value'], df['Vs_m_s'])[0, 1] ** 2
ax4.set_title(f'N vs Vs (R² = {r_sq:.2f})')

# Panel 5: Depth vs N scatter by soil type
ax5 = fig.add_subplot(3, 3, 5)
sns.scatterplot(data=df, x='depth_m', y='N_value', hue='soil_type',
                style='soil_type', s=60, alpha=0.7, ax=ax5)
ax5.set_title('Depth vs N-value')
ax5.set_xlabel('Depth (m)')
ax5.legend(fontsize=7, title_fontsize=8)

# Panel 6: Void ratio vs unit weight
ax6 = fig.add_subplot(3, 3, 6)
sns.regplot(data=df, x='void_ratio', y='unit_weight', ci=95,
            scatter_kws={'alpha': 0.5, 's': 30}, ax=ax6)
r_sq2 = np.corrcoef(df['void_ratio'], df['unit_weight'])[0, 1] ** 2
ax6.set_title(f'Void Ratio vs Unit Weight (R² = {r_sq2:.2f})')

# Panel 7-9: Correlation heatmap (spanning bottom row)
ax7 = fig.add_subplot(3, 1, 3)
corr_mat = df[numeric_cols].corr().round(2)
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax7)
ax7.set_title('Correlation Matrix', fontsize=13)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

print("This dashboard gives a complete statistical picture of the site.")
print("An engineer can quickly assess soil variability, correlations,")
print("and property ranges across all soil types.")

---

## Practice Exercises

### Exercise 1: Swarm Plot
Create a `sns.swarmplot()` showing individual N-value data points for each soil type. Overlay it on a boxplot using the same axes. Which soil type has the most variability?

### Exercise 2: Custom Heatmap
Create a heatmap of the **mean** soil properties (N_value, Vs, water_content, unit_weight) for each soil type. Use `df.groupby('soil_type').mean()` as the data source.

### Exercise 3: Joint Plot
Use `sns.jointplot()` with `kind='hex'` to show the density of N-value vs depth. Where are most data points concentrated?

In [ ]:
# EXERCISE 1
# fig, ax = plt.subplots(figsize=(8, 5))
# sns.boxplot(data=df, x='soil_type', y='N_value',
#             order=['Clay', 'Silt', 'Sand', 'Gravel'], ax=ax, fliersize=0)
# sns.swarmplot(data=df, x='soil_type', y='N_value',
#               order=['Clay', 'Silt', 'Sand', 'Gravel'],
#               color='black', size=4, ax=ax)
# ax.set_title('N-value: Box + Swarm')
# plt.show()

In [ ]:
# EXERCISE 2
# means = df.groupby('soil_type')[['N_value', 'Vs_m_s', 'water_content', 'unit_weight']].mean()
# fig, ax = plt.subplots(figsize=(8, 4))
# sns.heatmap(means.round(1), annot=True, fmt='.1f', cmap='YlOrRd', ax=ax)
# ax.set_title('Mean Soil Properties by Type')
# plt.show()

In [ ]:
# EXERCISE 3
# g = sns.jointplot(data=df, x='depth_m', y='N_value', kind='hex',
#                   cmap='YlOrBr', height=6)
# g.set_axis_labels('Depth (m)', 'SPT N-value')
# plt.show()

---

## Key Takeaways

1. **`histplot()`** with `kde=True` reveals the distribution shape — check if N-values are normally distributed before assuming Gaussian statistics in reports.

2. **`boxplot()`** instantly compares soil types. The box shows IQR, whiskers show range, and outliers flag unusual test results that need review.

3. **`heatmap()`** on a correlation matrix is the fastest way to find which soil properties are related. Strong N-vs-Vs correlation validates your empirical models.

4. **`pairplot()`** is your first step in exploratory data analysis — it shows all pairwise relationships in one figure, colored by soil type.

5. **`regplot()`** fits and visualizes regression with confidence intervals. Use FacetGrid to fit separate regressions per soil type for soil-specific correlations.

---

## How This Connects to ML

| Today's Concept | ML Equivalent |
|----------------|---------------|
| `histplot()` | Feature distribution check before modeling |
| `boxplot()` by group | Class-wise feature analysis for classification |
| `violinplot()` | Checking normality assumptions for Gaussian models |
| `heatmap()` correlation | Feature selection — drop highly correlated features |
| `pairplot()` | Exploratory Data Analysis (EDA) — the ML first step |
| `regplot()` | Visualizing linear model fit and residuals |
| `FacetGrid` | Comparing model performance across subgroups |
| Theme/styling | Making ML results presentable for stakeholders |

Seaborn is the go-to tool for the **EDA phase** of any ML project. Before building a model, you need to understand your data — and Seaborn makes that visual exploration effortless.

---

*Day 025 of 100 — Statistical plots turn numbers into geotechnical understanding!*